# CSAKD YOLO26s — Standalone Kaggle / Colab

**Eksperimen 08** · Adaptasi CSAKD enam modul untuk transfer YOLO26m → YOLO26s, dengan Cross Supervising, AAWM, dan AGKA.

Protokol penelitian: Poltrada V3, 100 epoch, imgsz640, batch16, seed42, AdamW lr0.001, AMP aktif. Teacher adalah best.pt dari artifact eksperimen teacher **v50** yang ditetapkan di section1; nomor versi artifact tidak menyatakan nomor epoch.

Notebook ini merupakan **adaptasi penelitian**, bukan reproduksi identik self-distillation YOLOv8 dan hybrid quantization dalam paper. Perbedaan formulasi paper/kode penulis dan batas pengujian dijelaskan di section5 dan laporan audit. Bobot AGKA/Cross 1/1 adalah pilihan adaptasi; belum diklaim optimal.

Jalankan smoke test sebelum eksperimen final. Evaluasi akhir menggunakan **validation (val)**; test tidak digunakan untuk tuning atau memilih checkpoint. Artifact per epoch menyediakan student-only best.pt/last.pt untuk inferensi, serta training_state_* asli yang memerlukan definisi class notebook. Resume CSAKD tidak didukung.


## 1. Experiment Configuration

In [ ]:
# Ubah parameter di bagian ini saja untuk menjalankan eksperimen.
EXPERIMENT_ID = "08_csakd_yolo26s"
METHOD = "csakd"
DATA = "/kaggle/input/datasets/ilhamrafiqin/poltrada-v3-yolo26-clean/data.yaml"  # Dataset bersih di Kaggle.
MODEL_WEIGHTS = "yolo26s.pt"
EPOCHS = 100
IMGSZ = 640
BATCH = 16
SEED = 42
DEVICE = 0                   # GPU Kaggle; 'cpu' untuk diagnosis lokal
WORKERS = 2
OPTIMIZER = "AdamW"
LR = 0.001
PATIENCE = 100
AMP = True

WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 3.0
CLOSE_MOSAIC = 10
EVAL_SPLIT = "val"           # validation; test tidak dipakai untuk pemilihan checkpoint
PROJECT = "/kaggle/working/results"  # Colab: ganti ke /content/results
TEACHER_CHECKPOINT = ""   # Diisi otomatis dari artifact W&B v50 di section 4.
KD_AGKA_WEIGHT = 1.0
KD_CROSS_WEIGHT = 1.0
EXPECTED_NC = 6
TEACHER_ARTIFACT = (
    "ilhamrafiqin-brawijaya-university/yolom26-kd-indotraffic/"
    "01_teacher_yolo26m-fux5wdcb-checkpoints:v50"
)
TEACHER_RUN_ID = "fux5wdcb"
CSAKD_AUTHOR_COMMIT = "3433559ab6c9cfdef3b4ac099f1bb2feddd42dc8"
DIAGNOSTIC_ONLY = False      # True hanya untuk fixture/smoke; bukan hasil penelitian.

# W&B online: setiap epoch mengunggah last.pt + best.pt sampai epoch tersebut.
WANDB_ENABLED = True
WANDB_PROJECT = "yolom26-kd-indotraffic"
WANDB_ENTITY = None          # gunakan akun default; ganti nama tim bila perlu


## 2. Environment Setup

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys
import os

# Pertahankan instalasi PyTorch/CUDA bawaan Kaggle; instal hanya yang dibutuhkan.
for package, version in (("ultralytics", "8.4.155"), ("wandb", "0.28.1")):
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        print(f"Installing {package}=={version} (was {installed})")
        subprocess.check_call([sys.executable, "-m", "pip", "install", f"{package}=={version}"])

import json
import time
from datetime import datetime
from pathlib import Path

import torch
import wandb
from ultralytics import YOLO, settings
from ultralytics.utils.torch_utils import init_seeds

# Satu sumber logging W&B: callback eksplisit di section 6, bukan integrasi otomatis Ultralytics.
settings.update({"wandb": False})
init_seeds(SEED, deterministic=True)
RUNTIME_VERSIONS = {
    "python": sys.version.split()[0], "torch": torch.__version__,
    "ultralytics": metadata.version("ultralytics"), "wandb": metadata.version("wandb"),
    "cuda_runtime": torch.version.cuda,
}
import ultralytics
if ultralytics.__version__ != "8.4.155" or wandb.__version__ != "0.28.1":
    raise RuntimeError("Versi paket yang sudah diimport berbeda dari pin; restart kernel setelah instalasi.")
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | Ultralytics {metadata.version('ultralytics')} | W&B {metadata.version('wandb')}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")
if DEVICE != "cpu" and not torch.cuda.is_available():
    raise RuntimeError("GPU tidak tersedia. Aktifkan GPU Kaggle/Colab atau set DEVICE='cpu' untuk test lokal.")

if WANDB_ENABLED:
    # Kaggle: tambahkan secret WANDB_API_KEY melalui Add-ons > Secrets dan aktifkan notebook access.
    if not os.environ.get("WANDB_API_KEY"):
        try:
            from kaggle_secrets import UserSecretsClient
            secret = UserSecretsClient().get_secret("WANDB_API_KEY")
            if secret:
                os.environ["WANDB_API_KEY"] = secret
        except ImportError:
            pass  # Colab atau lokal: sediakan WANDB_API_KEY sebagai environment variable.
    if not os.environ.get("WANDB_API_KEY"):
        raise RuntimeError("W&B online membutuhkan WANDB_API_KEY. Set sebagai Kaggle Secret atau environment variable; jangan tulis token ke notebook.")
    wandb.login(key=os.environ["WANDB_API_KEY"])


## 3. Dataset Loading

In [ ]:
from ultralytics.data.utils import check_det_dataset

# Memakai loader resmi, tanpa audit/cleaning dataset dalam notebook eksperimen.
data_info = check_det_dataset(DATA)
assert int(data_info["nc"]) > 0, "Dataset tidak memiliki kelas deteksi."
assert data_info.get("train") and data_info.get("val"), "Dataset wajib punya train dan val."
if EVAL_SPLIT != "val":
    raise ValueError("Protokol CSAKD menggunakan validation; test tidak boleh memilih checkpoint.")

# Periksa kesesuaian nama kelas teacher/dataset untuk metode KD.
def class_names(names):
    if isinstance(names, dict):
        return [str(names[k]) for k in sorted(names, key=lambda k: int(k))]
    return [str(x) for x in names]

print(f"Dataset: {DATA} | classes: {data_info['nc']} | validation: {EVAL_SPLIT}")

## 4. Model Initialization

In [ ]:
import base64
import hashlib
import warnings

# Alias eksplisit v50; tidak beralih ke latest atau teacher pretrained.
api = wandb.Api()
artifact = api.artifact(TEACHER_ARTIFACT, type="model")
if artifact.version != "v50":
    raise ValueError(f"Versi teacher harus v50, diterima: {artifact.version}")
artifact_metadata = dict(artifact.metadata or {})
producer_info, producer_config, provenance_missing = {}, {}, []
try:
    producer = artifact.logged_by()
    if producer is not None:
        producer_info = {"id": producer.id, "name": producer.name, "path": list(producer.path)}
        producer_config = dict(producer.config or {})
        if producer.id != TEACHER_RUN_ID:
            raise ValueError(f"Producer teacher bukan run yang ditetapkan: {producer.id}")
    else:
        provenance_missing.append("producer")
except ValueError:
    raise
except (wandb.errors.Error, RuntimeError, AttributeError) as error:
    provenance_missing.append(f"producer unavailable: {type(error).__name__}")

if "best.pt" not in artifact.manifest.entries:
    raise ValueError("Manifest artifact teacher tidak berisi best.pt; file cache lokal tidak boleh menggantikannya.")
teacher_entry = artifact.manifest.entries["best.pt"]
artifact_dir = Path(artifact.download(root="/kaggle/working/teacher_yolo26m_v50"))
teacher_path = artifact_dir / "best.pt"
if not teacher_path.is_file():
    raise FileNotFoundError(f"Artifact teacher tidak memiliki best.pt: {teacher_path}")
TEACHER_CHECKPOINT = str(teacher_path)
teacher_bytes = teacher_path.read_bytes()
teacher_sha256 = hashlib.sha256(teacher_bytes).hexdigest()
if teacher_entry.size is not None and len(teacher_bytes) != teacher_entry.size:
    raise ValueError("Ukuran best.pt berbeda dari manifest artifact.")
try:
    manifest_md5 = base64.b64decode(teacher_entry.digest, validate=True)
except (ValueError, TypeError):
    manifest_md5 = b""
if len(manifest_md5) == 16 and hashlib.md5(teacher_bytes).digest() != manifest_md5:
    raise ValueError("Checksum best.pt berbeda dari manifest artifact.")
del teacher_bytes

# Periksa checkpoint hasil eksperimen, kelas, dan skala arsitektur aktual.
teacher_meta = YOLO(str(teacher_path))
if int(data_info["nc"]) != EXPECTED_NC or int(teacher_meta.model.nc) != EXPECTED_NC:
    raise ValueError("Teacher dan Poltrada V3 harus memiliki enam kelas deteksi.")
if class_names(teacher_meta.model.names) != class_names(data_info["names"]):
    raise ValueError("Urutan nama kelas teacher dan dataset berbeda.")
architecture_yaml = str(teacher_meta.model.yaml.get("yaml_file", ""))
architecture_name = Path(architecture_yaml.replace("\\", "/").rsplit("/", 1)[-1]).stem
if architecture_name not in {"yolo26", "yolo26m"} or teacher_meta.model.yaml.get("scale") != "m":
    raise ValueError("Checkpoint teacher harus berasal dari keluarga YOLO26 dan memakai skala m.")
checkpoint_args = dict((teacher_meta.ckpt or {}).get("train_args") or {})

def verify_teacher_declarations(metadata_sources):
    """Tolak deklarasi berbeda; metadata kosong tidak dianggap verified."""
    declared_datasets, declared_models = {}, {}
    for source, values in metadata_sources.items():
        dataset = values.get("dataset", values.get("data"))
        if dataset:
            declared_datasets[source] = str(dataset)
            if str(dataset).replace("\\", "/") != str(DATA).replace("\\", "/"):
                raise ValueError(f"Dataset teacher berbeda ({source}): {dataset}")
        architecture = values.get("model")
        if architecture:
            declared_models[source] = str(architecture)
            basename = str(architecture).replace("\\", "/").rsplit("/", 1)[-1]
            if Path(basename).stem != "yolo26m":
                raise ValueError(f"Arsitektur teacher berbeda ({source}): {architecture}")
    return declared_datasets, declared_models

teacher_datasets, teacher_models = verify_teacher_declarations({
    "artifact_metadata": artifact_metadata,
    "producer_config": producer_config,
    "checkpoint_train_args": checkpoint_args,
})
if not teacher_datasets:
    provenance_missing.append("declared dataset")
if not teacher_models:
    provenance_missing.append("declared model")
if not artifact.digest:
    provenance_missing.append("artifact digest")
TEACHER_PROVENANCE = {
    "requested_artifact": TEACHER_ARTIFACT, "resolved_name": artifact.name,
    "version": artifact.version, "digest": artifact.digest,
    "best_pt_manifest_digest": teacher_entry.digest,
    "producer": producer_info,
    "producer_config": {key: producer_config[key] for key in (
        "experiment", "model", "dataset", "seed", "epochs", "imgsz", "batch",
        "optimizer", "learning_rate", "amp") if key in producer_config},
    "artifact_metadata": artifact_metadata, "best_pt_sha256": teacher_sha256,
    "checkpoint_path": str(teacher_path), "declared_datasets": teacher_datasets,
    "declared_models": teacher_models, "nc": int(teacher_meta.model.nc),
    "names": class_names(teacher_meta.model.names), "architecture_scale": "m",
    "architecture_yaml": architecture_yaml,
    "metadata_complete": not provenance_missing, "missing_metadata": provenance_missing,
}
if provenance_missing:
    warnings.warn("Provenance teacher belum lengkap: " + "; ".join(provenance_missing))
    if not DIAGNOSTIC_ONLY:
        raise ValueError("Provenance teacher belum lengkap; lengkapi bukti source/dataset sebelum run penelitian.")
print("Teacher:", teacher_path, "| SHA256:", teacher_sha256)
print("Artifact:", artifact.name, "| version:", artifact.version, "| producer:", producer_info)
print("Epoch metadata (bukan versi artifact):", artifact_metadata.get("epoch"))
del teacher_meta

model = YOLO(MODEL_WEIGHTS)
print("Student / model:", MODEL_WEIGHTS)


## 5. Knowledge Distillation Method

**Adaptasi CSAKD enam modul YOLO26m → s/n, bukan reproduksi identik paper.** Tap [4, 6, 10, 16, 19, 22] berada pada backbone/neck. Input/output student diproyeksikan ke channel teacher; cross-forward menjalankan modul teacher yang frozen terhadap input student. Resolusi spasial wajib sama; tidak ada resize diam-diam. Target teacher asli disimpan melalui detach().clone() sebelum cross-forward.

Acuan primer: [paper BMVC2025, Zhan dkk.](https://bmva-archive.org.uk/bmvc/2025/assets/papers/Paper_398/paper.pdf) dan [kode loss resmi pada commit3433559](https://github.com/KefanZhan/YOLOv8-KD/blob/3433559ab6c9cfdef3b4ac099f1bb2feddd42dc8/OtherKD/CSAKD.py).

**Formulasi numerik mengikuti kode penulis.** Dengan s output student terproyeksi, t snapshot teacher, c output cross, dan C channel: A=softmax(ReLU(c) @ ReLU(t).transpose(-1,-2) / sqrt(C), dim=-1) berbentuk **B×C×H×H**. AGKA=MSE(Norm_C((s+A@s)/2), Norm_C(t)). w=1-sigmoid(mean_B(sum_C(GAP(t)*GAP(s))/sqrt(C))); Cross=w*MSE(Norm_C(s), Norm_C(c)). Normalisasi L2 pada channel (eps=1e-12), MSE mean atas BCHW, KD FP32; jumlahkan keenam modul dengan bobot AGKA/Cross1/1.

**Perbedaan yang harus dilaporkan:** paper menyatakan Q dari teacher/K dari cross dan skala di luar softmax, sedangkan kode penulis menggunakan Q cross/K teacher dan skala sebelum softmax. Kode penulis menambahkan normalisasi channel/reduksi MSE mean. Notebook mengikuti kode untuk bagian numerik, dengan enam tap dan routing cross modul YOLO26 sebagai adaptasi (kode penulis memilih blok dalam cakupan layer berbeda). Paper membahas self-distillation YOLOv8 dan hybrid quantization; notebook memakai teacher YOLO26m/student s/n tanpa quantization. Bobot1/1 tidak diklaim optimal paper.

Teacher tetap frozen/eval. Jalur teacher dari gambar tanpa gradien; cross-forward tetap berada dalam autograd agar student dan projector menerima gradien. Detection loss end-to-end YOLO26 (one-to-many/one-to-one beserta jadwal criterion) dipertahankan. KD mean dikalikan batch size tepat sekali; item logging dan rasio KD/detection tidak ditambahkan lagi ke objective.


In [ ]:
import math
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
from ultralytics.nn.modules.head import Detect
from ultralytics.nn.distill_model import DistillationModel, FeatureHook
from ultralytics.models.yolo.detect import DetectionTrainer


def detect_head(network):
    heads = [m for m in network.model if isinstance(m, Detect)]
    if len(heads) != 1:
        raise ValueError("Adaptasi KD ini mensyaratkan tepat satu Detect head.")
    return heads[0]


def align_channels(cin, cout):
    return nn.Identity() if cin == cout else nn.Conv2d(cin, cout, 1, bias=False)


def snapshot_student_checkpoint(source, destination):
    """Save a portable student copy; never strip the live training checkpoint."""
    from ultralytics.utils.torch_utils import strip_optimizer

    source, destination = Path(source), Path(destination)
    if source.resolve() == destination.resolve():
        raise ValueError("Snapshot student harus berbeda dari checkpoint training asli.")
    if not source.is_file():
        raise FileNotFoundError(source)
    destination.parent.mkdir(parents=True, exist_ok=True)
    checkpoint = strip_optimizer(str(source), str(destination))
    student = checkpoint.get("model")
    if not isinstance(student, nn.Module) or isinstance(student, DistillationModel):
        raise RuntimeError("Gagal menghasilkan checkpoint student-only yang portabel.")
    if hasattr(student, "teacher_model") or hasattr(student, "projector"):
        raise RuntimeError("Checkpoint inference masih membawa modul distillation.")
    for module in student.modules():
        hooks = list(module._forward_hooks.values()) + list(module._forward_pre_hooks.values())
        if any(isinstance(hook, (FeatureHook, InputHook)) for hook in hooks):
            raise RuntimeError("Checkpoint student masih membawa feature hook training.")
    if not destination.is_file():
        raise RuntimeError("File snapshot student tidak dibuat.")
    return destination


def combined_loss(wrapper, batch, preds, weights):
    """Native detection objective + KD mean * B; diagnostics never enter the objective."""
    wrapper._teacher_feats.clear()
    wrapper._student_feats.clear()
    try:
        if not wrapper.training:
            base, items = wrapper.student_model.loss(batch, preds)
            kd_parts = {key: base.new_zeros(()) for key in weights}
        else:
            with torch.no_grad():
                wrapper.teacher_model(batch["img"])
            predictions = wrapper.student_model(batch["img"])
            base, items = wrapper.student_model.loss(batch, predictions)
            with torch.autocast(device_type=batch["img"].device.type, enabled=False):
                kd_parts = wrapper.kd_loss()
        kd = sum(weights[key] * kd_parts[key] for key in weights)
        if not torch.isfinite(base).all() or not all(torch.isfinite(value).all() for value in kd_parts.values()) or not torch.isfinite(kd).all():
            raise RuntimeError("Detection/KD loss non-finite; proses dihentikan.")
        batch_size = batch["img"].shape[0]
        detection_total = base.sum().detach() / batch_size
        items.update({key: value.detach() for key, value in kd_parts.items()})
        items["dis_loss"] = kd.detach()
        # E2ELoss native items report the one2one branch; this is the actual combined objective.
        items["det_total_loss"] = detection_total
        items["kd_to_det_ratio"] = kd.detach() / detection_total.abs().clamp_min(1e-12)
        return torch.cat((base.reshape(-1), (kd * batch_size).reshape(1))), items
    finally:
        wrapper._teacher_feats.clear()
        wrapper._student_feats.clear()


In [ ]:
def module_indices(network):
    # Six selected YOLO26 modules: backbone lateral outputs and neck Detect inputs.
    boundary = len(network.yaml["backbone"])
    lateral = set()
    for module in network.model[boundary:]:
        refs = module.f if isinstance(module.f, list) else [module.f]
        lateral.update(index for index in refs if 0 <= index < boundary)
    neck = list(detect_head(network).f)
    indices = sorted(lateral) + neck
    if indices != [4, 6, 10, 16, 19, 22]:
        raise ValueError("Adaptasi CSAKD ini memerlukan graph YOLO26 dengan tap [4, 6, 10, 16, 19, 22].")
    return indices


class InputHook:
    def __init__(self, cache, key):
        self.cache, self.key = cache, key

    def __call__(self, module, inputs):
        if len(inputs) != 1 or not isinstance(inputs[0], torch.Tensor):
            raise ValueError("Input modul CSAKD harus satu tensor.")
        self.cache[self.key] = inputs[0]


def csakd_losses(student, teacher, cross):
    """Numerical loss from author commit 3433559; row attention B,C,H,H, not HW,HW."""
    if student.ndim != 4 or student.shape != teacher.shape or teacher.shape != cross.shape:
        raise ValueError("Fitur CSAKD harus BCHW dan seukuran setelah channel projection.")
    s, t, c = student.float(), teacher.detach().float(), cross.float()
    scale = math.sqrt(s.shape[1])
    # Author code uses Q=cross, K=original teacher and scales logits before softmax.
    attention = torch.softmax(c.relu() @ t.relu().transpose(-2, -1) / scale, dim=-1)
    guidance = F.normalize((s + attention @ s) / 2, p=2, dim=1, eps=1e-12)
    target = F.normalize(t, p=2, dim=1, eps=1e-12)
    agka = F.mse_loss(guidance, target, reduction="mean")
    similarity = (t.mean((2, 3)) * s.mean((2, 3))).sum(dim=1) / scale
    weight = 1 - similarity.mean().sigmoid()  # one adaptive weight per module/batch
    discrepancy = F.mse_loss(
        F.normalize(s, p=2, dim=1, eps=1e-12),
        F.normalize(c, p=2, dim=1, eps=1e-12),
        reduction="mean",
    )
    return agka, weight * discrepancy


class CSAKDModel(DistillationModel):
    @staticmethod
    def get_distill_layers(network):
        return module_indices(network) + [detect_head(network).i]

    def _register_feature_hooks(self):
        super()._register_feature_hooks()
        for network, cache, handles in (
            (self.student_model, self._student_feats, self._student_hooks),
            (self.teacher_model, self._teacher_feats, self._teacher_hooks),
        ):
            if network is None:
                continue
            if self.feats_idx[:-1] != module_indices(network):
                raise ValueError("Index modul teacher/student CSAKD berbeda.")
            for index in self.feats_idx[:-1]:
                module = network.model[index]
                for key, hook in list(module._forward_pre_hooks.items()):
                    if isinstance(hook, InputHook):
                        del module._forward_pre_hooks[key]
                handles.append(module.register_forward_pre_hook(InputHook(cache, ("input", index))))

    def __init__(self, teacher_model, student_model):
        super().__init__(teacher_model, student_model)
        inputs, outputs = [], []
        for index in self.feats_idx[:-1]:
            sf, tf = self._student_feats[("input", index)], self._teacher_feats[("input", index)]
            so, to = self._student_feats[index], self._teacher_feats[index]
            if sf.shape[-2:] != tf.shape[-2:] or so.shape[-2:] != to.shape[-2:]:
                raise ValueError("Spatial feature teacher/student tidak seukuran.")
            inputs.append(align_channels(sf.shape[1], tf.shape[1]))
            outputs.append(align_channels(so.shape[1], to.shape[1]))
        self.projector = nn.ModuleDict({"input": nn.ModuleList(inputs), "output": nn.ModuleList(outputs)}).to(
            next(student_model.parameters()).device)
        self._teacher_feats.clear()
        self._student_feats.clear()

    def kd_loss(self):
        # Snapshot ALL original-image targets before ANY teacher module is called again.
        targets = {index: self._teacher_feats[index].detach().clone() for index in self.feats_idx[:-1]}
        components = []
        for pos, index in enumerate(self.feats_idx[:-1]):
            student = self.projector["output"][pos](self._student_feats[index].float())
            incoming = self.projector["input"][pos](self._student_feats[("input", index)].float())
            cross = self.teacher_model.model[index](incoming)  # frozen weights, autograd to student/input adapter
            components.append(csakd_losses(student, targets[index], cross))
        return {
            "kd_agka_loss": sum(x[0] for x in components),
            "kd_cross_loss": sum(x[1] for x in components),
        }

    def loss(self, batch, preds=None):
        return combined_loss(self, batch, preds, {"kd_agka_loss": KD_AGKA_WEIGHT, "kd_cross_loss": KD_CROSS_WEIGHT})


class KDTrainer(DetectionTrainer):
    def set_model_attributes(self):
        super().set_model_attributes()
        if self.args.compile or self.args.resume:
            raise ValueError("Adaptasi CSAKD: compile/resume tidak didukung.")
        self._requested_batch_size = self.args.batch
        self.model = CSAKDModel(self.args.distill_model, self.model)

    def _build_train_pipeline(self):
        # Ultralytics may halve batch after OOM; a changed protocol must stop, including retries.
        requested = getattr(self, "_requested_batch_size", self.args.batch)
        if self.batch_size != requested:
            raise RuntimeError(f"Batch CSAKD berubah dari {requested} ke {self.batch_size}; hentikan run dan evaluasi memori tanpa mengubah protokol diam-diam.")
        return super()._build_train_pipeline()

    def build_optimizer(self, model, name="auto", lr=0.001, momentum=0.9, decay=1e-5, iterations=1e5):
        optimizer = super().build_optimizer(model, name, lr, momentum, decay, iterations)
        for group in optimizer.param_groups:
            group["params"] = [parameter for parameter in group["params"] if parameter.requires_grad]
        expected = {id(parameter) for parameter in model.parameters() if parameter.requires_grad}
        actual = {id(parameter) for group in optimizer.param_groups for parameter in group["params"]}
        if expected != actual:
            raise RuntimeError("Optimizer CSAKD tidak mencakup seluruh parameter trainable.")
        return optimizer

    def train(self):
        try:
            return super().train()
        finally:
            from ultralytics.utils.torch_utils import unwrap_model
            net = unwrap_model(self.model)
            if hasattr(net, "_remove_feature_hooks"):
                net._remove_feature_hooks()


KD_TRAINER = KDTrainer
KD_TRAIN_ARGS = {"distill_model": str(teacher_path), "dis": 1.0, "compile": False}
KD_CONFIG = {
    "agka_weight": KD_AGKA_WEIGHT, "cross_weight": KD_CROSS_WEIGHT,
    "weight_policy": "fixed-study-adaptation-1-1", "taps": [4, 6, 10, 16, 19, 22],
    "loss_reference_commit": "3433559ab6c9cfdef3b4ac099f1bb2feddd42dc8",
    "loss_formulation": "author-normalized-mse-row-attention",
    "feature_normalization": {"p": 2, "dim": 1, "eps": 1e-12},
    "attention_shape": "B,C,H,H", "cross_routing": "six-local-student-input-modules",
    "kd_precision": "float32", "batch_scaling": "mean-times-batch-once",
}


## 6. Model Training

In [ ]:
from datetime import datetime

RUN_NAME = f"{EXPERIMENT_ID}_seed{SEED}_{datetime.now():%Y%m%d_%H%M%S}"
train_args = dict(
    data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    optimizer=OPTIMIZER, lr0=LR, seed=SEED, device=DEVICE,
    weight_decay=WEIGHT_DECAY, warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
    project=PROJECT, name=RUN_NAME, exist_ok=False,
    workers=WORKERS, patience=PATIENCE, plots=True,
    amp=AMP, pretrained=True, save=True,
)
train_args.update(KD_TRAIN_ARGS)

# W&B: satu run dan satu koleksi artifact checkpoint berversi per eksperimen.
wb_run = None
wb_state = {"uploaded_epochs": set(), "logged_epochs": set(), "best_epoch": None}
effective_protocol = {}

def record_effective_protocol(trainer):
    """Catat protokol aktual dan hentikan perubahan batch atau penonaktifan CUDA AMP."""
    actual_batch = int(trainer.batch_size)
    amp_active = bool(trainer.amp)
    effective_protocol.update({
        "requested_batch": int(BATCH), "effective_batch": actual_batch,
        "per_rank_loader_batch": int(trainer.train_loader.batch_size),
        "world_size": max(int(trainer.world_size), 1),
        "requested_amp": bool(AMP), "effective_amp": amp_active,
        "device_type": trainer.device.type,
        "diagnostic_only": bool(DIAGNOSTIC_ONLY),
    })
    if actual_batch != int(BATCH):
        raise RuntimeError("Batch efektif berubah; run dihentikan agar protokol penelitian tetap konsisten.")
    if not DIAGNOSTIC_ONLY and AMP and trainer.device.type == "cuda" and not amp_active:
        raise RuntimeError("CUDA AMP yang diminta tidak aktif; periksa kompatibilitas sebelum eksperimen final.")
    if wb_run is not None:
        wb_run.config.update({"effective_protocol": dict(effective_protocol)}, allow_val_change=True)

model.add_callback("on_train_start", record_effective_protocol)
model.add_callback("on_train_epoch_start", record_effective_protocol)
if WANDB_ENABLED:
    wb_run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=RUN_NAME,
        job_type="training",
        group="diagnostic" if DIAGNOSTIC_ONLY else "research",
        config={
            "experiment": EXPERIMENT_ID, "method": METHOD, "model": MODEL_WEIGHTS,
            "teacher_checkpoint": str(TEACHER_CHECKPOINT) if TEACHER_CHECKPOINT else None,
            "dataset": DATA, "eval_split": EVAL_SPLIT, "epochs": EPOCHS,
            "imgsz": IMGSZ, "batch": BATCH, "seed": SEED, "device": str(DEVICE),
            "optimizer": OPTIMIZER, "learning_rate": LR, "amp": AMP,
            "workers": WORKERS, "patience": PATIENCE,
            "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS,
            "close_mosaic": CLOSE_MOSAIC,
            "kd": KD_CONFIG, "diagnostic_only": DIAGNOSTIC_ONLY,
            "teacher_provenance": TEACHER_PROVENANCE, "runtime_versions": RUNTIME_VERSIONS,
            "csakd_author_commit": CSAKD_AUTHOR_COMMIT,
        },
        tags=[METHOD, EXPERIMENT_ID, "diagnostic" if DIAGNOSTIC_ONLY else "research"],
    )

    def save_epoch_artifact(trainer):
        """Snapshot checkpoint setiap epoch; upload selesai sebelum last.pt ditimpa epoch berikutnya."""
        epoch = int(trainer.epoch) + 1
        if epoch in wb_state["uploaded_epochs"]:
            return
        last_path = Path(trainer.last)
        best_path = Path(trainer.best)
        if not last_path.is_file():
            raise RuntimeError(f"Checkpoint epoch {epoch} tidak ada: {last_path}")
        fitness = None if trainer.fitness is None else float(trainer.fitness)
        best_fitness = None if trainer.best_fitness is None else float(trainer.best_fitness)
        is_best = fitness is not None and best_fitness is not None and fitness == best_fitness
        if is_best:
            wb_state["best_epoch"] = epoch
        artifact = wandb.Artifact(
            name=f"{EXPERIMENT_ID}-{wb_run.id}-checkpoints",
            type="model",
            description="Student-only best.pt/last.pt; checkpoint asli training_state_* memerlukan class notebook (resume tidak didukung).",
            metadata={
                "epoch": epoch, "fitness": fitness, "best_fitness": best_fitness,
                "is_best_epoch": bool(is_best), "best_epoch_so_far": wb_state["best_epoch"],
                "experiment": EXPERIMENT_ID, "dataset": DATA, "seed": SEED,
                "teacher_provenance": TEACHER_PROVENANCE, "runtime_versions": RUNTIME_VERSIONS,
                "kd": KD_CONFIG, "effective_protocol": dict(effective_protocol),
                "diagnostic_only": bool(DIAGNOSTIC_ONLY),
            },
        )
        snapshot_dir = Path(trainer.save_dir) / "student_checkpoints"
        snapshot_dir.mkdir(parents=True, exist_ok=True)
        portable_last = snapshot_student_checkpoint(last_path, snapshot_dir / "last.pt")
        artifact.add_file(str(portable_last), name="last.pt")
        artifact.add_file(str(last_path), name="training_state_last.pt")
        if best_path.is_file():
            portable_best = snapshot_student_checkpoint(best_path, snapshot_dir / "best.pt")
            artifact.add_file(str(portable_best), name="best.pt")
            artifact.add_file(str(best_path), name="training_state_best.pt")
        for filename in ("results.csv", "args.yaml"):
            file_path = Path(trainer.save_dir) / filename
            if file_path.is_file():
                artifact.add_file(str(file_path), name=filename)
        aliases = [f"epoch-{epoch:03d}", "latest"]
        if is_best:
            aliases.append("best")
        # Synchronous wait menjamin isi artifact epoch ini selesai dikirim sebelum file lokal berubah.
        wb_run.log_artifact(artifact, aliases=aliases).wait()
        wb_state["uploaded_epochs"].add(epoch)
        print(f"W&B checkpoint: epoch={epoch}, best={is_best}, aliases={aliases}")

    def log_epoch_metrics(trainer):
        """Satu record grafik per epoch (training loss, KD loss, validation, LR, best epoch)."""
        epoch = int(trainer.epoch) + 1
        # Ultralytics memanggil on_fit_epoch_end sekali lagi pada final_eval; abaikan panggilan ekstra.
        if epoch not in wb_state["uploaded_epochs"] or epoch in wb_state["logged_epochs"]:
            return
        loss = trainer.label_loss_items(trainer.tloss, prefix="train")
        payload = {"epoch": epoch, **{k: float(v) for k, v in loss.items()},
                   **{k: float(v) for k, v in trainer.lr.items()}}
        for key, value in trainer.metrics.items():
            try:
                payload[key] = float(value)
            except (ValueError, TypeError):
                continue
        payload["protocol/effective_batch"] = int(trainer.batch_size)
        payload["protocol/effective_amp"] = int(bool(trainer.amp))
        payload["checkpoint/is_best_epoch"] = int(wb_state["best_epoch"] == epoch)
        payload["checkpoint/best_epoch_so_far"] = wb_state["best_epoch"] or 0
        wb_run.log(payload, step=epoch)
        wb_state["logged_epochs"].add(epoch)

    model.add_callback("on_model_save", save_epoch_artifact)
    model.add_callback("on_fit_epoch_end", log_epoch_metrics)

print("Training:", EXPERIMENT_ID, "| data:", DATA, "| epochs:", EPOCHS,
      "| W&B:", "online + every epoch artifact" if WANDB_ENABLED else "disabled")
start_time = time.perf_counter()
try:
    if KD_TRAINER is None:
        model.train(**train_args)
    else:
        model.train(trainer=KD_TRAINER, **train_args)
except BaseException:
    if wb_run is not None:
        wb_run.finish(exit_code=1)
    raise
training_seconds = time.perf_counter() - start_time
run_dir = Path(model.trainer.save_dir)
best_pt = run_dir / "weights" / "best.pt"
if not best_pt.is_file():
    raise FileNotFoundError(f"Checkpoint best.pt tidak ditemukan: {best_pt}")
if wb_run is not None and len(wb_state["uploaded_epochs"]) != EPOCHS:
    print("Catatan: jumlah epoch selesai berbeda dari target (misalnya early stopping).",
          len(wb_state["uploaded_epochs"]), "/", EPOCHS)
print("Training selesai:", run_dir)


## 7. Validation & Reloaded Student Inference


In [ ]:
try:
    # Salinan student-only tidak mengubah checkpoint training asli.
    deployment_best_pt = snapshot_student_checkpoint(best_pt, run_dir / "student_checkpoints" / "best.pt")
    deployment_last_pt = snapshot_student_checkpoint(run_dir / "weights" / "last.pt", run_dir / "student_checkpoints" / "last.pt")
    best_model = YOLO(str(deployment_best_pt))
    metrics = best_model.val(
        data=DATA, split=EVAL_SPLIT, imgsz=IMGSZ, batch=BATCH,
        device=DEVICE, plots=True, verbose=False,
    )

    # Inference checkpoint reload pada satu gambar validation; box kosong tetap valid.
    inference_source = model.trainer.test_loader.dataset.im_files[0]
    reload_predictions = best_model.predict(source=inference_source, imgsz=IMGSZ, device=DEVICE, verbose=False)
    if len(reload_predictions) != 1 or reload_predictions[0].boxes is None:
        raise RuntimeError("Inferensi student reload tidak menghasilkan struktur deteksi yang valid.")
    if not torch.isfinite(reload_predictions[0].boxes.data).all():
        raise RuntimeError("Prediksi student reload mengandung NaN/Inf.")
    print("Validation selesai; inference student reload valid:", inference_source)

    from ultralytics.utils.torch_utils import get_flops
    parameters = sum(p.numel() for p in best_model.model.parameters())
    try:
        gflops = float(get_flops(best_model.model, imgsz=IMGSZ))
    except (AttributeError, ImportError, RuntimeError, TypeError, ValueError) as error:
        print("GFLOPs tidak tersedia:", error)
        gflops = None

    results = {
        "experiment": EXPERIMENT_ID,
        "method": METHOD,
        "model": MODEL_WEIGHTS,
        "dataset": DATA,
        "eval_split": EVAL_SPLIT,
        "evaluation_label": "validation", "diagnostic_only": bool(DIAGNOSTIC_ONLY),
        "seed": SEED,
        "epochs": EPOCHS,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "optimizer": OPTIMIZER,
        "learning_rate": LR,
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map),
        "parameters": int(parameters),
        "gflops": gflops,
        "inference_ms_per_image": float(metrics.speed.get("inference", 0.0)),
        "training_seconds": float(training_seconds),
        "device": str(DEVICE),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "teacher_checkpoint": str(TEACHER_CHECKPOINT) if METHOD != "none" else None,
        "best_pt": str(best_pt), "deployment_best_pt": str(deployment_best_pt),
        "deployment_last_pt": str(deployment_last_pt),
        "reload_inference_valid": True, "reload_inference_source": str(inference_source),
        "teacher_provenance": TEACHER_PROVENANCE, "runtime_versions": RUNTIME_VERSIONS,
        "effective_protocol": dict(effective_protocol),
    }
    if torch.cuda.is_available():
        results["peak_gpu_memory_gb"] = float(torch.cuda.max_memory_allocated() / 1024**3)
    print(json.dumps(results, indent=2, ensure_ascii=False))
except BaseException:
    if wb_run is not None:
        wb_run.finish(exit_code=1)
    raise


## 8. Results & Outputs

In [ ]:
# File lokal tetap tersimpan oleh Ultralytics, W&B menyimpan versi artifact tiap epoch.
run_dir.mkdir(parents=True, exist_ok=True)
results["best_epoch"] = wb_state["best_epoch"] if WANDB_ENABLED else None
(run_dir / "metrics.json").write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
configuration = {
    "experiment": EXPERIMENT_ID, "dataset": DATA, "evaluation_split": EVAL_SPLIT,
    "model": MODEL_WEIGHTS, "teacher": TEACHER_CHECKPOINT if METHOD != "none" else None,
    "seed": SEED, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH,
    "optimizer": OPTIMIZER, "lr0": LR, "device": DEVICE,
    "workers": WORKERS, "patience": PATIENCE, "amp": AMP,
    "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS,
    "close_mosaic": CLOSE_MOSAIC,
    "kd_hyperparameters": KD_CONFIG, "csakd_author_commit": CSAKD_AUTHOR_COMMIT,
    "teacher_provenance": TEACHER_PROVENANCE, "runtime_versions": RUNTIME_VERSIONS,
    "effective_protocol": dict(effective_protocol), "diagnostic_only": bool(DIAGNOSTIC_ONLY),
    "wandb_enabled": WANDB_ENABLED, "wandb_project": WANDB_PROJECT,
}
(run_dir / "config.json").write_text(json.dumps(configuration, indent=2, ensure_ascii=False), encoding="utf-8")

if wb_run is not None:
    summary = {f"final/{key}": val for key, val in results.items()
               if isinstance(val, (int, float)) and not isinstance(val, bool) and val is not None}
    wb_run.log(summary)
    wb_run.summary.update(summary)
    wb_run.summary["final/evaluation_split"] = EVAL_SPLIT
    wb_run.summary["final/evaluation_label"] = "validation"
    wb_run.summary["teacher_provenance_complete"] = TEACHER_PROVENANCE["metadata_complete"]
    wb_run.summary["best_epoch"] = wb_state["best_epoch"]
    wb_run.summary["checkpoint_collection"] = f"{EXPERIMENT_ID}-{wb_run.id}-checkpoints"
    wb_run.summary["uploaded_checkpoint_epochs"] = len(wb_state["uploaded_epochs"])

    # Ringkasan/config dan grafik sebagai Artifact terpisah, tanpa menggandakan checkpoint model.
    final_artifact = wandb.Artifact(
        name=f"{EXPERIMENT_ID}-{wb_run.id}-results", type="results",
        metadata={"experiment": EXPERIMENT_ID, "best_epoch": wb_state["best_epoch"]},
    )
    for filename in ("metrics.json", "config.json", "results.csv", "args.yaml"):
        candidate = run_dir / filename
        if candidate.is_file():
            final_artifact.add_file(str(candidate), name=filename)
    for folder_name, folder in (("training", run_dir),
                                ("evaluation", Path(getattr(metrics, "save_dir", run_dir)))):
        if folder.exists():
            for plot_path in sorted(folder.glob("*.png")):
                key = f"plots/{folder_name}/{plot_path.stem}"
                wb_run.log({key: wandb.Image(str(plot_path))})
                final_artifact.add_file(str(plot_path), name=f"{folder_name}/{plot_path.name}")
    wb_run.log_artifact(final_artifact, aliases=["final"]).wait()
    print("W&B:", wb_run.url)
    print("Checkpoint alias: epoch-001 ... epoch-%03d, latest, best" % len(wb_state["uploaded_epochs"]))
    wb_run.finish()

print("Hasil lokal:", run_dir)
print("  - weights/best.pt, weights/last.pt (student-only setelah final_eval Ultralytics)")
if WANDB_ENABLED:
    print("  - Artifact per epoch: training_state_best.pt/last.pt (state asli, class notebook diperlukan)")
print("  - student_checkpoints/best.pt, last.pt (student-only untuk inferensi)")
print("  - results.csv, args.yaml")
print("  - metrics.json, config.json")
